In [0]:
catalog = "_exponent"

silver_schema = "omop_silver"
mapping_schema = "omop_mapping"

gold_schemas = [
    "omop_epic"
]

source_systems = [
    "omop_epic"
]

exclude_gold_tables = [
    "concept",
    "concept_ancestor",
    "concept_class",
    "concept_relationship",
    "concept_synonym",
    "domain",
    "drug_strength",
    "relationship",
    "source_to_concept_map",
    "vocabulary"
]

source_system_list = ", ".join([f"'{source_system}'" for source_system in source_systems])
gold_schema_list = ", ".join([f"'{schema}'" for schema in gold_schemas])
exclude_gold_table_list = ", ".join([f"'{table}'" for table in exclude_gold_tables])


silver_tables = spark.sql(f"""
  SELECT DISTINCT table_name
  FROM {catalog}.information_schema.columns
  WHERE table_schema = '{silver_schema}'
    AND column_name = 'source_system'
  ORDER BY table_name
""").collect()

mapping_tables = spark.sql(f"""
  SELECT DISTINCT table_name
  FROM {catalog}.information_schema.columns
  WHERE table_schema = '{mapping_schema}'
    AND column_name = 'source_system'
  ORDER BY table_name
""").collect()

gold_tables = spark.sql(f"""
  SELECT table_schema, table_name
  FROM {catalog}.information_schema.tables
  WHERE table_catalog = '{catalog}'
    AND table_schema IN ({gold_schema_list})
    AND table_type IN ('MANAGED', 'EXTERNAL')
    AND data_source_format = 'DELTA'
    AND table_name NOT IN ({exclude_gold_table_list})
  ORDER BY table_schema, table_name
""").collect()


print("Deleting silver records...")
for row in silver_tables:
    table_name = row["table_name"]

    delete_sql = f"""
      DELETE FROM {catalog}.{silver_schema}.{table_name}
      WHERE source_system IN ({source_system_list})
    """

    print(delete_sql.strip())
    spark.sql(delete_sql)


print("Deleting mapping records...")
for row in mapping_tables:
    table_name = row["table_name"]

    delete_sql = f"""
      DELETE FROM {catalog}.{mapping_schema}.{table_name}
      WHERE source_system IN ({source_system_list})
    """

    print(delete_sql.strip())
    spark.sql(delete_sql)


print("Truncating gold tables...")
for row in gold_tables:
    table_schema = row["table_schema"]
    table_name = row["table_name"]

    truncate_sql = f"""
      TRUNCATE TABLE {catalog}.{table_schema}.{table_name}
    """

    print(truncate_sql.strip())
    spark.sql(truncate_sql)